# RAG with LangChain: DataCamp Chapter 1

This notebook reproduces the exercises from [Retrieval Augmented Generation (RAG) with LangChain](https://app.datacamp.com/learn/courses/retrieval-augmented-generation-rag-with-langchain), Chapter 1. 


## Loading PDF Files

Paper downloaded from [[2005.11401] Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://arxiv.org/abs/2005.11401)

In [1]:
# LangChain interface to `pypdf`
from langchain_community.document_loaders import PyPDFLoader

pdf_loader = PyPDFLoader('rag_paper.pdf')
documents = pdf_loader.load()

ModuleNotFoundError: No module named 'langchain_community'

In [ ]:
# perhaps each page is a document?
len(documents)

In [ ]:
# attributes include `metadata` and `page_content`
documents[0]

LangChain has similar interfaces for CSV, HTML, and probably others. 

## Text splitting, embeddings, and vector storage

Splitting the text into chunks, retrieving embeddings for them from OpenAI, and storing them in Chroma. 

In [ ]:
# define a character-based text splitter
from langchain_text_splitters import CharacterTextSplitter

text = """Machine learning is a fascinating field.\n\nIt involves algorithms and models that
can learn from data. These models can then make predictions or decisions without being
explicitly programmed to perform the task.\nThis capability is increasingly valuable in
various industries, from finance to healthcare.\n\nThere are many types of machine learning,
including supervised, unsupervised, and reinforcement learning.\nEach type has its own
strengths and applications.
"""

text_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=100,
    chunk_overlap=10,
)

chunks = text_splitter.split_text(text)
for chunk in chunks:
    print(f"{len(chunk)}: {chunk!r}")


Some chunks above are too big. A `RecursiveTextSplitter` can create chunks that conform to the maximum size by recursively applying additional separators. 

I'm not sure what the implications are for varying chunk sizes, or overlap. 

Note two chunks still have newlines: why?

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],
    chunk_size=100,
    chunk_overlap=10
)

chunks = splitter.split_text(text)
for chunk in chunks:
    print(f"{len(chunk)}: {chunk!r}")


In [ ]:
# now applying this to the PDF documents
# documents is defined above
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# split_documents handles multiple documents: split_text only handles one
chunks = splitter.split_documents(documents)

for chunk in chunks[:3]:
    print(f"{len(chunk.page_content)}: {chunk!r}")

In [ ]:
# now convert the chunks to embeddings with OpenAI and store in Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# my API key on Biblica's account: keep private
openai_ai_key = "sk-proj-DxrI2tc54TPpENkBpc7sT3BlbkFJLt3n7tUegFVZsX8f3F0m"
embedding_model = OpenAIEmbeddings(
    api_key=openai_ai_key,
    model="text-embedding-3-small"
)
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)
